# Arbitrage Strategy Example

This notebook demonstrates how to use arbitrage strategies to identify and execute profitable opportunities across exchanges.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import asyncio

from src.strategies.arbitrage_strategy import (
    ArbitrageStrategy,
    ArbitrageType,
    ArbitrageOpportunity
)
from src.data.binance_fetcher import BinanceFetcher

## Configuration

Load and configure arbitrage strategy settings.

In [ ]:
# Load configuration
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Initialize strategy
strategy = ArbitrageStrategy(
    config=config,
    exchanges=['binance', 'kraken', 'coinbase'],
    min_profit_threshold=0.001,
    max_position_size=1.0
)

print("Strategy initialized with exchanges:")
for exchange in strategy.exchanges:
    print(f"- {exchange}")

## Market Data

Fetch and analyze market data from different exchanges.

In [ ]:
async def fetch_market_data():
    """Fetch market data from exchanges."""
    symbol = 'BTC/USDT'
    orderbooks = await strategy.fetch_orderbooks(symbol)
    
    print("\nOrderbook Summary:")
    for exchange, book in orderbooks.items():
        best_bid = max(book['bids'].keys())
        best_ask = min(book['asks'].keys())
        spread = (best_ask - best_bid) / best_bid * 100
        
        print(f"\n{exchange}:")
        print(f"Best Bid: ${best_bid:,.2f}")
        print(f"Best Ask: ${best_ask:,.2f}")
        print(f"Spread: {spread:.4f}%")
    
    return orderbooks

# Fetch data
orderbooks = await fetch_market_data()

## Opportunity Analysis

Analyze arbitrage opportunities across exchanges.

In [ ]:
def analyze_opportunities(orderbooks: dict):
    """Analyze arbitrage opportunities."""
    # Identify opportunities
    opportunities = strategy.identify_opportunities(
        orderbooks,
        volume=0.1
    )
    
    # Filter opportunities
    filtered = strategy.filter_opportunities(opportunities)
    
    print(f"\nFound {len(opportunities)} opportunities")
    print(f"Filtered to {len(filtered)} viable opportunities")
    
    if filtered:
        print("\nTop Opportunities:")
        for opp in sorted(
            filtered,
            key=lambda x: x.net_profit,
            reverse=True
        )[:3]:
            print(f"\n{opp.exchange_a} -> {opp.exchange_b}:")
            print(f"Entry: ${opp.price_a:,.2f}")
            print(f"Exit: ${opp.price_b:,.2f}")
            print(f"Spread: ${opp.spread:,.2f}")
            print(f"Net Profit: ${opp.net_profit:,.2f}")
    
    return filtered

# Analyze opportunities
opportunities = analyze_opportunities(orderbooks)

## Risk Analysis

Analyze risks associated with arbitrage opportunities.

In [ ]:
def visualize_risks(opportunities: list):
    """Visualize opportunity risks."""
    if not opportunities:
        print("No opportunities to analyze")
        return
    
    # Extract risk metrics
    risk_data = pd.DataFrame([
        {
            'opportunity': f"{opp.exchange_a}->{opp.exchange_b}",
            **opp.risk_metrics
        }
        for opp in opportunities
    ])
    
    # Plot risk metrics
    plt.figure(figsize=(15, 5))
    
    # Risk metrics heatmap
    plt.subplot(1, 2, 1)
    risk_metrics = risk_data.drop('opportunity', axis=1)
    sns.heatmap(
        risk_metrics.T,
        cmap='YlOrRd',
        xticklabels=risk_data['opportunity'],
        annot=True,
        fmt='.3f'
    )
    plt.title('Risk Metrics Heatmap')
    plt.xticks(rotation=45)
    
    # Risk-reward scatter
    plt.subplot(1, 2, 2)
    plt.scatter(
        [opp.net_profit for opp in opportunities],
        [sum(opp.risk_metrics.values()) for opp in opportunities]
    )
    plt.xlabel('Net Profit')
    plt.ylabel('Total Risk')
    plt.title('Risk-Reward Analysis')
    
    plt.tight_layout()
    plt.show()

# Visualize risks
visualize_risks(opportunities)

## Execution Simulation

Simulate arbitrage execution with paper trading.

In [ ]:
async def simulate_execution(opportunities: list):
    """Simulate arbitrage execution."""
    results = []
    
    for opp in opportunities:
        # Execute arbitrage
        success = await strategy.execute_arbitrage(opp)
        
        if success:
            results.append({
                'timestamp': datetime.fromtimestamp(opp.timestamp),
                'exchange_pair': f"{opp.exchange_a}->{opp.exchange_b}",
                'entry_price': opp.price_a,
                'exit_price': opp.price_b,
                'volume': opp.volume_a,
                'profit': opp.net_profit
            })
    
    return pd.DataFrame(results)

# Simulate execution
results_df = await simulate_execution(opportunities)

if not results_df.empty:
    print("Execution Results:")
    print(f"Total Trades: {len(results_df)}")
    print(f"Total Profit: ${results_df['profit'].sum():,.2f}")
    print(f"Average Profit: ${results_df['profit'].mean():,.2f}")
    
    # Plot results
    plt.figure(figsize=(15, 5))
    
    # Cumulative profit
    plt.subplot(1, 2, 1)
    plt.plot(results_df['timestamp'],
             results_df['profit'].cumsum())
    plt.title('Cumulative Profit')
    plt.xticks(rotation=45)
    
    # Profit distribution
    plt.subplot(1, 2, 2)
    sns.histplot(results_df['profit'])
    plt.title('Profit Distribution')
    
    plt.tight_layout()
    plt.show()

## Performance Analysis

Analyze strategy performance metrics.

In [ ]:
def analyze_performance():
    """Analyze strategy performance."""
    metrics = strategy.get_performance_metrics()
    
    print("Performance Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
    
    # Plot performance metrics
    plt.figure(figsize=(15, 5))
    
    # Metrics by exchange pair
    if not results_df.empty:
        plt.subplot(1, 2, 1)
        exchange_metrics = results_df.groupby('exchange_pair')[
            'profit'
        ].agg(['mean', 'std', 'count'])
        exchange_metrics.plot(kind='bar')
        plt.title('Performance by Exchange Pair')
        plt.xticks(rotation=45)
    
    # Active positions
    plt.subplot(1, 2, 2)
    positions = pd.Series(strategy.active_positions).value_counts()
    positions.plot(kind='pie')
    plt.title('Active Positions by Exchange')
    
    plt.tight_layout()
    plt.show()

# Analyze performance
analyze_performance()

## Live Monitoring

Monitor live arbitrage opportunities.

In [ ]:
async def monitor_opportunities(duration: int = 60):
    """Monitor live arbitrage opportunities."""
    start_time = time.time()
    monitoring_data = []
    
    while time.time() - start_time < duration:
        # Fetch orderbooks
        orderbooks = await strategy.fetch_orderbooks('BTC/USDT')
        
        # Identify opportunities
        opportunities = strategy.identify_opportunities(
            orderbooks,
            volume=0.1
        )
        
        # Record data
        monitoring_data.append({
            'timestamp': datetime.now(),
            'num_opportunities': len(opportunities),
            'max_profit': max([o.net_profit for o in opportunities])
            if opportunities else 0
        })
        
        # Update plot
        if len(monitoring_data) % 5 == 0:
            df = pd.DataFrame(monitoring_data)
            
            plt.clf()
            plt.figure(figsize=(15, 5))
            
            plt.subplot(1, 2, 1)
            plt.plot(df['timestamp'], df['num_opportunities'])
            plt.title('Number of Opportunities')
            plt.xticks(rotation=45)
            
            plt.subplot(1, 2, 2)
            plt.plot(df['timestamp'], df['max_profit'])
            plt.title('Maximum Profit')
            plt.xticks(rotation=45)
            
            plt.tight_layout()
            plt.show()
        
        await asyncio.sleep(1)
    
    return pd.DataFrame(monitoring_data)

# Monitor opportunities
print("Monitoring opportunities for 60 seconds...")
monitoring_df = await monitor_opportunities()

print("\nMonitoring Summary:")
print(f"Average opportunities: {monitoring_df['num_opportunities'].mean():.2f}")
print(f"Maximum profit seen: ${monitoring_df['max_profit'].max():,.2f}")